# Using the Pelican Client

OSPool Guest Notebooks ([notebook.ospool.osg-htc.org](notebook.ospool.osg-htc.org)) come with the Pelican CLI Client pre-installed.
Otherwise, you can download the CLI client from [docs.pelicanplatform.org/install/linux-binary](docs.pelicanplatform.org/install/linux-binary). 

> The Pelican software is bundled with the HTCondor Software Suite.
> If you have HTCondor, then you should also have the Pelican CLI Client as well.
> Behind the scenes, HTCondor uses the Pelican "Plugin" Client to manage transfers of files declared in the submit file using the `osdf://` or `pelican://` prefixes.

The Pelican CLI Client is accessible via the `pelican` command in the terminal.
There are a variety of subcommands, but for this tutorial we'll focus on the `pelican object` command.

In [ ]:
# Show the help text for the command
pelican object --help

## Listing objects

You can use the command 

```
pelican object ls <OSDF_URL>
```

to `ls` the object in the OSDF federation. 
This is useful for confirming the existence of the object. 

In [ ]:
# Confirm that "hello-world.txt" exists
pelican object ls osdf:///pelicanplatform/test/hello-world.txt

In [ ]:
# List the objects contained within the namespace prefix "/pelicanplatform/test"
pelican object ls osdf:///pelicanplatform/test

In [ ]:
# List the properties of objects contained within the namespace prefix "/pelicanplatform/test"
pelican object ls -l osdf:///pelicanplatform/test

## Getting public objects

If the object is publically-readable (meaning no authentication is required to access the object), then getting said object using the Pelican client is similar to the `cp` command.

```
pelican object get <OSDF_URL> <local_destination>
```

Like with the `cp` command, a destination must be provided.
Similarly, you can use a different name for the object at the local destination.

In [ ]:
# Download an object from the OSDF
pelican object get osdf:///pelicanplatform/test/hello-world.txt ./
ls

In [ ]:
# Download and rename an object from the OSDF
pelican object get osdf:///pelicanplatform/test/hello-world.txt ./my-local-object.txt
ls

## Modifying transfers with HTML `?` queries

Normally, HTML queries are appended to the end of URLs with the beginning marked by a `?`. 
This can be used to modify the request and reply to the web server.
Pelican similarly uses this syntax to modify how objects are transferred.

### `?recursive`

When this option is appended to the end of a URL for a `get` request, it requests that all of the objects with the same prefix are downloaded.

```
pelican object get <OSDF_URL>?recursive
```

The objects will be saved to a directory with the same basename as the provided path, unless a different directory name is provided.

In [ ]:
# Download the contents of "/pelicanplatform/test" to the current directory, automatically saved to directory "test" (basename of provided URL)
pelican object get osdf:///pelicanplatform/test?recursive ./
ls

In [ ]:
ls test

In [ ]:
# Download the contents of "/pelicanplatform/test" to the current directory under the custom-named "my-local-objects" directory
pelican object get osdf:///pelicanplatform/test?recursive ./my-local-objects
ls my-local-objects

The `?recursive` query can also be used when declaring file transfers in HTCondor submit files.
Submit the example `recursive.sub` HTCondor job to see it in practice.

In [ ]:
cat recursive.sub

In [ ]:
condor_submit recursive.sub

Use `condor_q` or `condor_watch_q` in your terminal tab to monitor the progress of the job.
When complete, check the contents of the corresponding `.out` and `.err` files.

In [ ]:
cat recursive.out

The output file should show a directory called `noaa` with the contents of the `/ospool/uc-shared/public/OSG-Staff/training/noaa` namespace prefix, 
which includes three `.csv` files and a `.tar.gz` file (which coincidentally is a compressed copy of the three `.csv` files).

### `?pack`

The `?pack` query requires a corresponding value, i.e., `?pack=<value>`. 
There are a handful of values that can be set.
Simplest value is `auto`.

Regardless of which value is selected, Pelican employs the following behavior

* **On Download**: If the listed object is compressed using the specified protocol (or any recognized when using `auto`), Pelican will try to automatically decompress the object on `get`.
* **On Upload**: Pelican will automatically compress the files into a single object when `put` into a namespace. (If using `auto`, defaults to `.tar.xz` compression.)

To demonstrate the utility, let's compare the data access method both with and without using the `?pack` option.

#### Normal, manual approach

In [ ]:
# Download the listed ".tar.gz" file normally
pelican object get osdf:///ospool/uc-shared/public/OSG-Staff/training/noaa/ghcnd-files.tar.gz .

In [ ]:
# Manually decompress file; creates "ghcnd-files" directory
tar -xzf ghcnd-files.tar.gz
ls ghcnd-files/

**Note:** For the manual approach, there must be enough disk space for holding both the original compressed file (`ghcnd-files.tar.gz`) and its decompressed contents (`ghcnd-files` directory and contents).

In [ ]:
# reset for other approach
rm -r ghcnd-files/* ghcnd-files.tar.gz
ls

#### Using `?pack=auto`

In [ ]:
# Download the listed ".tar.gz" file and decompress using the "?pack=auto" option
pelican object get osdf:///ospool/uc-shared/public/OSG-Staff/training/noaa/ghcnd-files.tar.gz?pack=auto .

In [ ]:
ls

In [ ]:
ls ghcnd-files

Not only is this shorter, but you only need enough disk space for the final object.

#### In HTCondor jobs

The `?pack` query can also be used when declaring file transfers in HTCondor submit files.
Submit the example `pack.sub` HTCondor job to see it in practice.

In [ ]:
cat pack.sub

In [ ]:
condor_submit pack.sub

Use `condor_q` or `condor_watch_q` in your terminal tab to monitor the progress of the job.
When complete, check the contents of the corresponding `.out` and `.err` files.

In [ ]:
cat pack.out

#### On Upload

As discussed in the next section, uploading data to create an object in the federation requires authentication, which we won't be able to test in this training format.

In principle, though, you would provide the directory of contents you want to upload locally and add the `?pack=<value>` query to the end of the destination OSDF URL:

```
pelican object put my-local-dir osdf:///<namespace_prefix>/<compressed_object_name>?pack=<value>
```

A similar outcome can be achieved for the output of an HTCondor job using declarations in the submit file:

```
transfer_output_files = output_directory
transfer_output_remaps = "output_directory = osdf:///<namespace_prefix>/<compressed_object_name>?pack=<value>
```

assuming that the necessary credential management features have been correctly configured for HTCondor,
and the submitting user is permitted access to the provided namespace prefix.

## Getting protected objects, writing objects

Objects in a namespace prefix can be accessed by anyone only if "public reads" is enabled for that namespace.
If the namespace does not have "public reads" enabled, then the objects in that namespace prefix can only be accessed if the client provides the necessary authorization token.

To write objects to a namespace prefix, two things are required:

1. The namespace prefix must be "write" enabled.
2. The client must provide the appropriate authorization token for writing to that namespace prefix.

The default namespace setup requires that the authorization token is generated at the issuer configured for that namespace prefix.
This can be difficult for the general user.

Pelican, however, can be integrated with CILogon and other OAuth2 providers for generating tokens. 
In that case, the client will automatically prompt the user to authenticate through a browser link and save the resulting token locally.
Then the user just runs their commands as normal.

### Example: getting protected object

Each user on an OSPool AP has access to a protected read-only namespace prefix in the OSDF, at `osdf:///ospool/apXX/data/<username>`.
Normal usage is for HTCondor jobs to download user's data from a corresponding directory on their AP via said namespace prefix,
with authentication mediated by the credential manager built into HTCondor.

In principle, the Pelican client can be used to download data via that namespace prefix as well, assuming that client can be authenticated by the same user account.
The process looks like the following (only tested with `ap40`) when using CILogon for the token issuer:

```
$ pelican object get osdf:///ospool/ap40/data/<username>/test.txt ./
The client is able to save the authorization in a local file.
This prevents the need to reinitialize the authorization for each transfer.
You will be asked for this password whenever a new session is started.
Please provide a new password to encrypt the local OSDF client configuration file:

The OSDF client configuration is encrypted.  Enter your password for the local OSDF client configuration file:

To approve credentials for this operation, please navigate to the following URL and approve the request:

https://osdf-ospool-issuer.osgdev.chtc.io/scitokens-server/device?user_code=WVG-HWX-3TH
```

The user opens the correspending URL in their web browser, where they'll see a page like this:

![Browser with prompt to login to CILogon for selected institution](images/pelican-CILogon-authentication-initial.png)

After authenticating through the appropriate institution, they'll see a page like this:

![Browser with confirmation that access has been approved](images/pelican-CILogon-authentication-confirmed.png)

If authentication is successful, a corresponding token will be created in the Pelican client, which it will then try to use
to access the object in the protected namespace.
Accessing said object will only be successful, however, if the generated token (and the user it represents) is permitted access
to the protected namespace.

Putting an object into a namespace is the exact same process, but using the command

```
pelican object put <local_data> osdf:///<namespace_prefix>/<object_name>
```